# Final Project

## Introduction
*What dataset are you looking at? Where/how was it created? What questions will be asked?*

In this project, we are analyzing the California Housing Prices dataset. The dataset contains median house values, population counts, number of rooms, and spatial/demographic features for various districts.

**Research Questions:**
1. How does the proximity to the ocean affect median house prices?
2. How does household income impact house prices?
3. Where are the most expensive houses located?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/housing.csv')
df.head()

## Cleaning and Organizing Data

First, we perform diagnostics on the dataset. We examine skewed values and missing data. We will visualize the distribution of missing data using a heatmap, followed by an imputation process. Additionally, we will analyze some outliers and handle them appropriately for the project.

In [ ]:
# 1. Visualize missing values
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title("Heatmap of Missing Values")
plt.show()

# Get a numerical summary:
missing_info = df.isnull().sum()
print("Missing Value Counts:\n", missing_info[missing_info > 0])

# `total_bedrooms` has missing values. To avoid significantly changing the characteristics,
# of the neighborhoods, we'll impute these missing values using the median.
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())

# 2. Outlier Analysis
# In the California housing dataset, `median_house_value` is frequently capped at 500,001.
# Let's analyze this situation:
capped_houses = df[df['median_house_value'] >= 500000]
print(f"\nNumber of houses valued at $500,000 and above (capped): {len(capped_houses)}")

# For analytical accuracy and future modeling, it is often preferable to remove these capped records.
# Since our dataset is large enough, we can prevent artificial outliers by filtering out 
# these artificially capped maximum-priced values.
df_clean = df[df['median_house_value'] < 500000].copy()

print(f"\nRow count before cleaning: {len(df)}")
print(f"Row count after cleaning (outliers/capped removed): {len(df_clean)}")

## Visualizations

In this section, we will create **4 different visualizations** focusing on house prices, geographical distribution, and the features of independent variables.

1. **Frequency Distribution of House Prices:** Examining how prices are distributed.
2. **Median Prices by Ocean Proximity:** Price distributions across categorical variables (Boxplot).
3. **Impact of Income Level on House Prices:** The linear relationship between household income and house prices (Scatterplot).
4. **Geographical Price Heatmap:** Mapping where the most expensive and most densely populated houses are located based on coordinates.

In [ ]:
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('California Housing Data - Exploratory Visualizations', fontsize=18)

# 1. Histogram of Median House Values
sns.histplot(df_clean['median_house_value'], kde=True, bins=50, ax=axes[0,0], color='skyblue')
axes[0,0].set_title("1. Distribution of Median House Values")
axes[0,0].set_xlabel("Median House Value ($)")
axes[0,0].set_ylabel("Frequency")

# 2. Boxplot: Ocean Proximity vs. House Value
sns.boxplot(x='ocean_proximity', y='median_house_value', data=df_clean, ax=axes[0,1], palette='Pastel1')
axes[0,1].set_title("2. House Prices by Ocean Proximity")
axes[0,1].set_xlabel("Ocean Proximity")
axes[0,1].set_ylabel("Median House Value ($)")

# 3. Scatter Plot: Median Income vs. House Value
sns.scatterplot(x='median_income', y='median_house_value', data=df_clean, alpha=0.3, ax=axes[1,0], color='coral')
axes[1,0].set_title("3. Impact of Median Income on House Prices")
axes[1,0].set_xlabel("Median Income (Tens of Thousands $)")
axes[1,0].set_ylabel("Median House Value ($)")

# 4. Geographical Data (Longitude vs Latitude colored by House Value)
scatter = axes[1,1].scatter(df_clean['longitude'], df_clean['latitude'], 
                            c=df_clean['median_house_value'], s=df_clean['population']/100, 
                            cmap='jet', alpha=0.5)
axes[1,1].set_title("4. Geographical Price Distribution & Population Density (Circle Size: Population)")
axes[1,1].set_xlabel("Longitude")
axes[1,1].set_ylabel("Latitude")
plt.colorbar(scatter, ax=axes[1,1], label='Median House Value ($)')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

**Visualization Interpretations:**
1. **Histogram:** After removing grouped/capped values, prices are mostly concentrated around $100,000 to $300,000, showing a right-skewed distribution.
2. **Boxplot:** `INLAND` areas have the lowest house prices, while `<1H OCEAN` and `NEAR OCEAN` houses see significant value increases. Houses on an `ISLAND` have the highest median prices.
3. **Scatter Plot:** As income levels increase, median house prices also tend to rise significantly. This visual indicates the presence of a strong positive linear relationship.
4. **Geographical Plot:** Along ranges parallel to the California coastline, especially in high-population areas (indicated by larger circles) like Los Angeles and the San Francisco Bay Area, houses are noticeably more expensive.

## Descriptive Analysis

In the descriptive analysis section, we will focus on numerically understanding the relationships between variables.

1. **Statistical Summary Table & Aggregation:** We will group the data by ocean proximity and compute the mean to observe the average of median house values and median incomes.
2. **Correlation Matrix:** Linear relationships between numerical features will be examined and interpreted (especially relationships with median house value).

In [ ]:
# 1. Aggregation / Groupby
summary_stats = df_clean.groupby('ocean_proximity')[['median_house_value', 'median_income']].mean().round(2).sort_values(by='median_house_value', ascending=False)
display(summary_stats)

print("\nInterpretation: Neighborhoods strictly on an ISLAND or near the ocean have both higher income levels and significantly higher average house prices.\nThe average house price for those living inland (INLAND) is severely lower, almost half of those near the ocean.\n")

# 2. Correlation Matrix
numeric_df = df_clean.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Matrix of Numerical Variables")
plt.show()

print("Correlation Interpretation:")
print("- There is a relatively high positive correlation (r=~0.64) between `median_income` and `median_house_value`.")
print("- There are very strong linear relationships (0.91+) between `total_rooms` and `total_bedrooms`, as well as between `population` and `households`, indicating a strong presence of multicollinearity.")

## Conclusions

In this study, an Exploratory Data Analysis (EDA) was conducted on the **California Housing** dataset. Our initial research questions have been answered as follows:

1. **Impact of Ocean Proximity on Prices:** Median house prices systematically increase as one gets closer to the ocean or descends towards the coastline. While Island communities in this dataset host the most expensive houses, they are numerically limited. Real metropolitan life proves there is great density in bay/coastal regions like San Francisco. Coastal proximity commands an evident premium, whereas inland areas hold the lowest house prices overall.
2. **Income & Price Linearity:** As expected, one of the most important factors dictating a house's price is the median income of the neighborhood's households (evidenced by the positive correlation with `median_income`).
3. **Geographical Location:** As clearly seen from our mapping visualization, housing in high population density coastal areas of California, such as Los Angeles and the Bay Area, are by far the most expensive real estate units in the state.

**Limitations:**
* Prices were "capped" at `$500,001` in the original dataset, meaning houses naturally more expensive than 500k were not accurately measured, skewing the right-tail distributions. We filtered this limit out to avoid skewed findings altogether.
* This exploratory analysis explains the fundamental variable relationships at an overview level, which can easily guide a future Machine Learning (e.g., Multiple Regression) modeling study. In future implementations, a complete price prediction model could be developed after resolving existing variables' multicollinearity issues (possibly with PCA).